In [ ]:
# Load Monte Carlo results of Methane Pyrolysis
import json
from pathlib import Path

# Find project root
try:
    start_dir = Path(__file__).resolve().parent
except NameError:
    start_dir = Path.cwd()

project_root = next(
    (p for p in [start_dir, *start_dir.parents]
     if all((p / folder).is_dir() for folder in ["FeMP", "ACMP", "NiMP"])),
    None
)

if project_root is None:
    raise FileNotFoundError("Project root not found.")

# Directories
dir_Fe = project_root / "FeMP" / "Results_TEA"
dir_AC = project_root / "ACMP" / "Results_TEA"
dir_Ni = project_root / "NiMP" / "Results_TEA"

# JSON files
file_mc_Fe = dir_Fe / "LCOH_MonteCarlo.json"
file_mc_AC = dir_AC / "LCOH_MonteCarlo.json"
file_mc_Ni = dir_Ni / "LCOH_MonteCarlo.json"

# Load JSON results
with open(file_mc_Fe, "r") as f: mc_Fe = json.load(f)
with open(file_mc_AC, "r") as f: mc_AC = json.load(f)
with open(file_mc_Ni, "r") as f: mc_Ni = json.load(f)

In [ ]:
# Load Monte Carlo results of SMR and Water Electrolysis
import json
from pathlib import Path

# Find project root
try:
    start_dir = Path(__file__).resolve().parent
except NameError:
    start_dir = Path.cwd()

project_root = next(
    (p for p in [start_dir, *start_dir.parents]
     if all((p / folder).is_dir() for folder in ["SMR", "WaterElectrolysis"])),
    None
)

if project_root is None:
    raise FileNotFoundError("Project root not found.")

# Directories
dir_SMR = project_root / "SMR"
dir_WE = project_root / "WaterElectrolysis"

# JSON files
file_mc_SMR = dir_SMR / "LCOH_MonteCarlo_SMR.json"
file_mc_WE = dir_WE / "LCOH_MonteCarlo_WE.json"

# Load JSON results
with open(file_mc_SMR, "r") as f: mc_SMR = json.load(f)
with open(file_mc_WE, "r") as f: mc_WE = json.load(f)


In [ ]:
# Monte Carlo methane pyrolysis only plot

import json
import numpy as np
import matplotlib.pyplot as plt
import matplotlib as mpl
import scienceplots

from pathlib import Path
from matplotlib.ticker import MultipleLocator

plt.style.use(["science", "ieee"])

tick_font = 14.0
axis_label_font = 16.0
legend_font = 13.0

mpl.rcParams.update({
    "text.usetex": True,
    "font.family": "serif",
    "font.serif": ["Times"],
    "font.size": tick_font,
    "axes.labelsize": axis_label_font,
    "xtick.labelsize": tick_font,
    "ytick.labelsize": tick_font,
    "legend.fontsize": legend_font,
    "figure.facecolor": "white",
    "axes.facecolor": "white",
    "savefig.facecolor": "white",
    "text.color": "black",
    "axes.labelcolor": "black",
    "axes.edgecolor": "black",
    "xtick.color": "black",
    "ytick.color": "black",
    "axes.grid": False,
    "xtick.direction": "in",
    "ytick.direction": "in",
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
})

# Load files

with open(file_mc_Ni, "r") as f:
    mc_Ni = json.load(f)

with open(file_mc_Fe, "r") as f:
    mc_Fe = json.load(f)

with open(file_mc_AC, "r") as f:
    mc_AC = json.load(f)

#Extract data

lcoh_Ni = np.asarray(mc_Ni["metrics"]["LCOP"], dtype=float)
lcoh_Fe = np.asarray(mc_Fe["metrics"]["LCOP"], dtype=float)
lcoh_AC = np.asarray(mc_AC["metrics"]["LCOP"], dtype=float)

#Reference smr and electrolysis

num_bins = 60

smr_min = 2.2
smr_max = 3.3

electrolysis_min = 4.8
electrolysis_max = 8.5

color_Ni = "#E69F00"
color_Fe = "#7B4AB5"
color_AC = "#D62728"

band_smr_color = "#AFCDEB"
band_ele_color = "#B4D8B4"

text_smr_color = "#1976D2"
text_ele_color = "#2E7D32"

def normal_pdf(x, mu, sigma):
    return (1.0 / (sigma * np.sqrt(2.0 * np.pi))) * np.exp(
        -0.5 * ((x - mu) / sigma) ** 2
    )

mu_Ni = np.mean(lcoh_Ni)
sigma_Ni = np.std(lcoh_Ni)

mu_Fe = np.mean(lcoh_Fe)
sigma_Fe = np.std(lcoh_Fe)

mu_AC = np.mean(lcoh_AC)
sigma_AC = np.std(lcoh_AC)

#print statistics

print("\n================ MONTE CARLO STATISTICS ================\n")

print(f"Nickel:          mu = {mu_Ni:.4f}, sigma = {sigma_Ni:.4f}")
print(f"Iron:            mu = {mu_Fe:.4f}, sigma = {sigma_Fe:.4f}")
print(f"Activated carbon: mu = {mu_AC:.4f}, sigma = {sigma_AC:.4f}")

all_data = np.concatenate([
    lcoh_Ni,
    lcoh_Fe,
    lcoh_AC
])

x_min = min(
    np.min(all_data),
    smr_min
) - 0.5

x_max = max(
    np.max(all_data),
    electrolysis_max
) + 0.5

bins = np.linspace(
    x_min,
    x_max,
    num_bins + 1
)

x_grid = np.linspace(
    x_min,
    x_max,
    1000
)


pdf_Ni = normal_pdf(
    x_grid,
    mu_Ni,
    sigma_Ni
)

pdf_Fe = normal_pdf(
    x_grid,
    mu_Fe,
    sigma_Fe
)

pdf_AC = normal_pdf(
    x_grid,
    mu_AC,
    sigma_AC
)


fig, ax = plt.subplots(
    figsize=(10.6, 6.8)
)

fig.patch.set_facecolor("white")
ax.set_facecolor("white")

ax.axvspan(
    smr_min,
    smr_max,
    facecolor=band_smr_color,
    alpha=0.70,
    edgecolor="none",
    zorder=0
)

ax.axvspan(
    electrolysis_min,
    electrolysis_max,
    facecolor=band_ele_color,
    alpha=0.70,
    edgecolor="none",
    zorder=0
)

hist_Ni = ax.hist(
    lcoh_Ni,
    bins=bins,
    density=True,
    color=color_Ni,
    alpha=0.42,
    edgecolor="white",
    linewidth=0.7,
    label="Nickel",
    zorder=2
)

hist_Fe = ax.hist(
    lcoh_Fe,
    bins=bins,
    density=True,
    color=color_Fe,
    alpha=0.34,
    edgecolor="white",
    linewidth=0.7,
    label="Iron",
    zorder=2
)

hist_AC = ax.hist(
    lcoh_AC,
    bins=bins,
    density=True,
    color=color_AC,
    alpha=0.34,
    edgecolor="white",
    linewidth=0.7,
    label="Activated carbon",
    zorder=2
)

# Normal fit curves
line_Ni, = ax.plot(
    x_grid,
    pdf_Ni,
    color=color_Ni,
    linewidth=2.5,
    linestyle="-",
    label=rf"Ni: $\mu={mu_Ni:.2f}$, $\sigma={sigma_Ni:.2f}$",
    zorder=4
)

line_Fe, = ax.plot(
    x_grid,
    pdf_Fe,
    color=color_Fe,
    linewidth=2.5,
    linestyle="-",
    label=rf"Fe: $\mu={mu_Fe:.2f}$, $\sigma={sigma_Fe:.2f}$",
    zorder=4
)

line_AC, = ax.plot(
    x_grid,
    pdf_AC,
    color=color_AC,
    linewidth=2.5,
    linestyle="-",
    label=rf"AC: $\mu={mu_AC:.2f}$, $\sigma={sigma_AC:.2f}$",
    zorder=4
)

ax.set_xlabel(
    r"Average LCOH / [US\$\,$\cdot$\,kg$^{\mbox{-}1}$]",
    fontsize=axis_label_font,
    labelpad=10
)

ax.set_ylabel(
    r"Probability density",
    fontsize=axis_label_font,
    labelpad=10
)

y_max_hist = max(
    np.max(hist_Ni[0]),
    np.max(hist_Fe[0]),
    np.max(hist_AC[0])
)

y_max_pdf = max(
    np.max(pdf_Ni),
    np.max(pdf_Fe),
    np.max(pdf_AC)
)

y_max_total = max(
    y_max_hist,
    y_max_pdf
)

ax.set_xlim(
    x_min,
    x_max
)

ax.set_ylim(
    0,
    y_max_total * 1.14
)

ax.yaxis.set_major_locator(
    MultipleLocator(0.025)
)

ax.yaxis.set_minor_locator(
    MultipleLocator(0.005)
)

y_text_top = y_max_total * 1.10
y_text_second = y_max_total * 1.04

ax.text(
    (smr_min + smr_max) / 2.0,
    y_text_top,
    r"\textbf{SMR}",
    color=text_smr_color,
    fontsize=14,
    ha="center",
    va="center"
)

ax.text(
    (electrolysis_min + electrolysis_max) / 2.0,
    y_text_second,
    r"\textbf{Electrolysis}",
    color=text_ele_color,
    fontsize=14,
    ha="center",
    va="center"
)

ax.xaxis.set_major_locator(
    MultipleLocator(5)
)

ax.xaxis.set_minor_locator(
    MultipleLocator(1)
)

ax.tick_params(
    axis="both",
    which="major",
    direction="in",
    top=True,
    bottom=True,
    left=True,
    right=True,
    length=8,
    width=1.2,
    labelsize=tick_font,
    colors="black"
)

ax.tick_params(
    axis="both",
    which="minor",
    direction="in",
    top=True,
    bottom=True,
    left=True,
    right=True,
    length=4,
    width=1.0,
    colors="black"
)

ax.grid(False)

for spine in ax.spines.values():
    spine.set_visible(True)
    spine.set_color("black")
    spine.set_linewidth(1.2)

handles = [
    hist_Ni[2][0],
    hist_Fe[2][0],
    hist_AC[2][0],
    line_Ni,
    line_Fe,
    line_AC
]

labels = [
    "Nickel",
    "Iron",
    "Activated carbon",
    rf"Ni: $\mu={mu_Ni:.2f}$, $\sigma={sigma_Ni:.2f}$",
    rf"Fe: $\mu={mu_Fe:.2f}$, $\sigma={sigma_Fe:.2f}$",
    rf"AC: $\mu={mu_AC:.2f}$, $\sigma={sigma_AC:.2f}$"
]

legend = ax.legend(
    handles,
    labels,
    loc="upper right",
    frameon=True,
    fancybox=False,
    framealpha=1.0,
    facecolor="white",
    edgecolor="0.50",
    fontsize=legend_font,
    handlelength=2.0,
    handletextpad=0.6,
    borderpad=0.45,
    labelspacing=0.35
)

legend.get_frame().set_linewidth(
    0.8
)

plt.tight_layout()


fig.savefig("monte_carlo_all_catalysts.pdf",
    format="pdf"
)

plt.show()

In [ ]:
#Monte Carlo all hydrogen technologies

import json
import numpy as np
import matplotlib.pyplot as plt
import matplotlib as mpl
import scienceplots
from matplotlib.ticker import MultipleLocator, MaxNLocator, AutoMinorLocator

plt.style.use(["science", "ieee"])

tick_font = 14.0
axis_label_font = 16.0
legend_font = 13.0

mpl.rcParams.update({
    "text.usetex": True,
    "font.family": "serif",
    "font.serif": ["Times"],
    "font.size": tick_font,
    "axes.labelsize": axis_label_font,
    "xtick.labelsize": tick_font,
    "ytick.labelsize": tick_font,
    "legend.fontsize": legend_font,
    "figure.facecolor": "white",
    "axes.facecolor": "white",
    "savefig.facecolor": "white",
    "text.color": "black",
    "axes.labelcolor": "black",
    "axes.edgecolor": "black",
    "xtick.color": "black",
    "ytick.color": "black",
    "axes.grid": False,
    "xtick.direction": "in",
    "ytick.direction": "in",
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
})

#Load results

with open(file_mc_Ni, "r") as f: mc_Ni = json.load(f)
with open(file_mc_Fe, "r") as f: mc_Fe = json.load(f)
with open(file_mc_AC, "r") as f: mc_AC = json.load(f)
with open(file_mc_SMR, "r") as f: mc_SMR = json.load(f)
with open(file_mc_WE, "r") as f: mc_WE = json.load(f)

#Extract distributions
lcoh_Ni = np.asarray(mc_Ni["metrics"]["LCOP"], dtype=float)
lcoh_Fe = np.asarray(mc_Fe["metrics"]["LCOP"], dtype=float)
lcoh_AC = np.asarray(mc_AC["metrics"]["LCOP"], dtype=float)
lcoh_SMR = np.asarray(mc_SMR["metrics"]["LCOP"], dtype=float)
lcoh_WE = np.asarray(mc_WE["metrics"]["LCOP"], dtype=float)

#Layout

num_bins = 60

color_Ni = "#E69F00"
color_Fe = "#7B4AB5"
color_AC = "#D62728"

band_smr_color = "#AFCDEB"
band_ele_color = "#B4D8B4"

text_smr_color = "#1976D2"
text_ele_color = "#2E7D32"

#Legend
technologies = {
    "SMR": {"data": lcoh_SMR, "fill": band_smr_color, "line": text_smr_color, "name": "SMR"},
    "WE": {"data": lcoh_WE, "fill": band_ele_color, "line": text_ele_color, "name": "Water electrolysis"},
    "Ni": {"data": lcoh_Ni, "fill": color_Ni, "line": color_Ni, "name": "Nickel"},
    "Fe": {"data": lcoh_Fe, "fill": color_Fe, "line": color_Fe, "name": "Iron"},
    "AC": {"data": lcoh_AC, "fill": color_AC, "line": color_AC, "name": "Activated carbon"},
}

#Print statistics

def normal_pdf(x, mu, sigma):
    return (1.0 / (sigma * np.sqrt(2.0 * np.pi))) * np.exp(-0.5 * ((x - mu) / sigma) ** 2)

print("\n================ MONTE CARLO STATISTICS ================\n")

for key, tech in technologies.items():
    data = tech["data"]
    if not np.all(np.isfinite(data)) or len(data) < 2:
        raise ValueError(f"Invalid Monte Carlo samples for {key}.")

    tech["mu"] = np.mean(data)
    tech["sigma"] = np.std(data)

    if tech["sigma"] <= 0:
        raise ValueError(f"Zero standard deviation for {key}.")

    print(f"{tech['name']:<20} mu = {tech['mu']:.4f}, sigma = {tech['sigma']:.4f}")


all_data = np.concatenate([tech["data"] for tech in technologies.values()])

x_min = np.min(all_data) - 0.5
x_max = np.max(all_data) + 0.5
x_grid = np.linspace(x_min, x_max, 3000)

for tech in technologies.values():
    tech["pdf"] = normal_pdf(x_grid, tech["mu"], tech["sigma"])

#Plot

fig, ax = plt.subplots(figsize=(10.6, 6.8))

fig.patch.set_facecolor("white")
ax.set_facecolor("white")

hist_handles = []
line_handles = []
hist_labels = []
line_labels = []
y_max_hist = 0.0
y_max_pdf = 0.0

# Histograms and normal fit curves
for key, tech in technologies.items():

    hist = ax.hist(
        tech["data"], bins=num_bins, density=True,
        color=tech["fill"], alpha=0.42 if key == "Ni" else 0.34,
        edgecolor="white", linewidth=0.7,
        label=tech["name"], zorder=2
    )

    line, = ax.plot(
        x_grid, tech["pdf"], color=tech["line"],
        linewidth=2.5, linestyle="-",
        label=rf"{key}: $\mu={tech['mu']:.2f}$, $\sigma={tech['sigma']:.2f}$",
        zorder=4
    )

    hist_handles.append(hist[2][0])
    line_handles.append(line)

    hist_labels.append(tech["name"])
    line_labels.append(
        rf"{key}: $\mu={tech['mu']:.2f}$, $\sigma={tech['sigma']:.2f}$"
    )

    y_max_hist = max(y_max_hist, np.max(hist[0]))
    y_max_pdf = max(y_max_pdf, np.max(tech["pdf"]))


ax.set_xlabel(
    r"Average LCOH / [US\$\,$\cdot$\,kg$^{\mbox{-}1}$]",
    fontsize=axis_label_font, labelpad=10
)

ax.set_ylabel(
    r"Probability density",
    fontsize=axis_label_font, labelpad=10
)

y_max_total = max(y_max_hist, y_max_pdf)

ax.set_xlim(x_min, x_max)
ax.set_ylim(0, y_max_total * 1.14)

ax.xaxis.set_major_locator(MultipleLocator(5))
ax.xaxis.set_minor_locator(MultipleLocator(1))

ax.yaxis.set_major_locator(MaxNLocator(nbins=6))
ax.yaxis.set_minor_locator(AutoMinorLocator(5))

ax.tick_params(
    axis="both", which="major", direction="in",
    top=True, bottom=True, left=True, right=True,
    length=8, width=1.2, labelsize=tick_font, colors="black"
)

ax.tick_params(
    axis="both", which="minor", direction="in",
    top=True, bottom=True, left=True, right=True,
    length=4, width=1.0, colors="black"
)

ax.grid(False)

for spine in ax.spines.values():
    spine.set_visible(True)
    spine.set_color("black")
    spine.set_linewidth(1.2)

handles = hist_handles + line_handles
labels = hist_labels + line_labels

legend = ax.legend(
    handles, labels,
    loc="upper right", ncol=2,
    frameon=True, fancybox=False,
    framealpha=1.0, facecolor="white",
    edgecolor="0.50", fontsize=legend_font,
    handlelength=2.0, handletextpad=0.6,
    borderpad=0.45, labelspacing=0.35,
    columnspacing=1.2
)

legend.get_frame().set_linewidth(0.8)

plt.tight_layout()

fig.savefig("monte_carlo_all_technologies.pdf", format="pdf")

plt.show()